# Solution 2.4: Transforming Data & Creating New Features (Angola IEA)

This notebook turns the cleaned file into an analysis ready dataset, and ends by
producing Angola's headline labour market statistic two different ways.

You will practice:
- Banding a continuous variable with `pd.cut()`
- Decoding numeric codes with `.map()` and a dictionary
- Building a three way category with `np.select()`
- Binary flags with `np.where()` and updates with `.loc[]`
- Chained, dependent columns with `assign()` and lambdas
- Custom row logic with `apply()`
- Weighting a statistic, and seeing why the definition matters more than the code

> **Pipeline:** run Exercise 2.3 first. Writes to `20_processed/`.

### Path Setup (run first)

In [1]:
import os

import numpy as np
import pandas as pd

DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'
clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

STR_COLS = {
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
}
df = pd.read_csv(clean_path, dtype=STR_COLS)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

Loaded: (53297, 27)


,household_id,cluster_id,province_code,rel_to_head,area_type,quarter,sex,person_no,age,marital_status,...,job_start_year,hours_usual,hours_actual,sought_work,sought_business,wants_work,available_now,available_2wk,interview_date,weight_ind
0,10000003,1000,25,1.00,2.00,4.00,1.00,1,55.00,3.00,...,NaN,NaN,NaN,2.00,2.00,2.00,NaN,NaN,2025-12-14,"1,399.39"
1,10000003,1000,25,2.00,2.00,4.00,2.00,2,52.00,3.00,...,NaN,NaN,NaN,2.00,2.00,2.00,NaN,NaN,2025-12-14,"1,253.95"
2,10000003,1000,25,3.00,2.00,4.00,1.00,3,19.00,1.00,...,NaN,NaN,NaN,2.00,2.00,2.00,NaN,NaN,2025-12-14,"1,495.07"
3,10000003,1000,25,3.00,2.00,4.00,1.00,4,16.00,1.00,...,NaN,NaN,NaN,1.00,NaN,NaN,1.00,NaN,2025-12-14,"1,495.07"
4,10000009,1000,25,1.00,2.00,4.00,2.00,1,43.00,6.00,...,NaN,NaN,NaN,2.00,2.00,2.00,NaN,NaN,2025-12-14,"1,253.95"


---

## Task 1: Age bands with `pd.cut()`

The labour statistics that follow all rest on the working age population, which
Angola defines as 15 and over. Band the ages accordingly.

Note `right=False`, which makes each interval closed on the left: a 15 year old
belongs to Youth, not Child.

In [2]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 15, 25, 65, 120],
    labels=['Child', 'Youth', 'Adult', 'Elderly'],
    right=False,
)
df['age_group'].value_counts(dropna=False)

age_group
Child      23526
Adult      17199
Youth      10890
Elderly     1682
Name: count, dtype: int64

**Answers:**

- Child 23,526, Youth 10,890, Adult 17,199, Elderly 1,682. They sum to 53,297,
  so nothing fell outside the bins.
- With the default `right=True` the intervals would close on the right, putting
  15 year olds in Child and quietly shrinking the working age population by
  everyone aged exactly 15.
- Children are 44% of this sample. That is the single most important fact about
  Angola's labour market and it is visible before any modelling.

---

## Task 2: Decode the numeric codes with `.map()`

The codebook read in 2.1 gives the code to label mapping. `.map()` applies a
dictionary element by element and returns `NaN` for anything not in it, which
doubles as a check.

In [3]:
PROVINCE_MAP = {
    '10': 'Cabinda', '11': 'Zaire', '12': 'Uíge', '13': 'Bengo', '14': 'Luanda',
    '15': 'Cuanza-Norte', '16': 'Cuanza-Sul', '17': 'Malanje', '18': 'Lunda-Norte',
    '19': 'Lunda-Sul', '20': 'Moxico', '21': 'Bié', '22': 'Huambo', '23': 'Benguela',
    '24': 'Namibe', '25': 'Huila', '26': 'Cunene', '27': 'Cubango',
    '28': 'Icolo e Bengo', '29': 'Moxico Leste', '30': 'Cuando',
}
SEX_MAP = {1: 'Masculino', 2: 'Feminino'}
AREA_MAP = {1: 'Urbana', 2: 'Rural'}
EDUCATION_MAP = {
    1: 'Primário', 2: 'I Ciclo Secundário', 3: 'II Ciclo Secundário',
    4: 'Bacharelato', 5: 'Licenciatura', 6: 'Mestrado', 7: 'Doutoramento',
    9: 'Nenhum nível',
}

df['province_name'] = df['province_code'].map(PROVINCE_MAP)
df['sex_label'] = df['sex'].map(SEX_MAP)
df['area_label'] = df['area_type'].map(AREA_MAP)
df['education_label'] = df['education_level'].map(EDUCATION_MAP)

print('Unmapped provinces:', df['province_name'].isna().sum())
print(df['area_label'].value_counts(dropna=False))
print()
print(df['education_label'].value_counts(dropna=False))

Unmapped provinces: 0
area_label
Urbana    34129
Rural     19168
Name: count, dtype: int64

education_label
NaN                    30144
Primário               12525
I Ciclo Secundário      5159
II Ciclo Secundário     4481
Licenciatura             780
Nenhum nível             133
Bacharelato               61
Mestrado                  13
Doutoramento               1
Name: count, dtype: int64


**Answers:**

- Zero unmapped provinces, because the dictionary was built from the file's own
  codebook rather than from memory.
- 34,129 urban and 19,168 rural.
- `education_label` is `NaN` for the majority of rows, because `education_level`
  is 56.6% missing: the question is only asked of people the questionnaire routes
  to it. `.map()` returns `NaN` both for a genuinely missing input and for a code
  absent from the dictionary, so the two causes look identical in the output. That
  is why the unmapped count above is worth printing separately.
- Note that the education codes run 1 to 7 and then jump to 9, with no 8. Code 9
  means "Nenhum nível", no level at all, so it is not a rank above 7. Sorting or
  averaging this column as if it were ordinal would put the least educated group
  at the top.
- The three most recent provinces, Icolo e Bengo, Moxico Leste and Cuando, were
  created in 2024. A mapping copied from an older publication would leave 6,499
  people, 12.2% of the sample, with a missing province name. 2.5 shows exactly
  that failure as a merge.

---

## Task 3: Labour force status with `np.select()`

`np.select()` takes conditions in order and applies the first match. This is the
heart of the notebook, and the definitions matter more than the syntax.

**Employed:** worked for pay, or worked on own account, or has a job they were
absent from. **Unemployed (strict ILO):** not employed, actively looked for work,
and available to start. Everyone else of working age is outside the labour force.

Note what this rule leaves out: `worked_family_business`, the flag for
contributing family workers, people who work unpaid in a household member's
farm or business. ICLS-19, the international labour statistics standard,
counts them as employed when the unit is a market one. Excluding them here is
not something the data forces on you: it is a definitional choice, made
silently unless you say so. Task 4 comes back to this and shows what the
number does when the choice is made differently.

In [4]:
working_age = df['age'] >= 15
employed = (
    (df['worked_for_pay'] == 1)
    | (df['worked_own_account'] == 1)
    | (df['absent_from_job'] == 1)
)
seeking = (df['sought_work'] == 1) | (df['sought_business'] == 1)
available = (df['available_now'] == 1) | (df['available_2wk'] == 1)

strict_conditions = [working_age & employed, working_age & seeking & available]
df['lf_status_strict'] = np.select(
    strict_conditions, ['Employed', 'Unemployed'], default='Outside labour force')

df['lf_status_strict'].value_counts()

lf_status_strict
Outside labour force    34968
Employed                15697
Unemployed               2632
Name: count, dtype: int64

In [5]:
# The relaxed definition also counts people who want work but have stopped
# looking: the discouraged, who a strict measure treats as economically inactive.
relaxed_conditions = [
    working_age & employed,
    working_age & ((seeking & available) | (df['wants_work'] == 1)),
]
df['lf_status_relaxed'] = np.select(
    relaxed_conditions, ['Employed', 'Unemployed'], default='Outside labour force')

df['lf_status_relaxed'].value_counts()

lf_status_relaxed
Outside labour force    29742
Employed                15697
Unemployed               7858
Name: count, dtype: int64

**Answers:**

- Strict: 15,697 employed, 2,632 unemployed, 34,968 outside the labour force.
- Relaxed: the same 15,697 employed, but 7,858 unemployed. Around 5,200 people
  move from "outside the labour force" to "unemployed" purely because the
  definition changed.
- `available_now` and `available_2wk` are combined with `|` because they are two
  stages of one question: `available_2wk` is only asked of people who answered no
  to `available_now`, which is why it is 98.5% missing. Using it alone would
  discard almost every unemployed person.
- Order matters. Employment is tested first, so somebody who is both working and
  looking for a better job counts as employed, which is the standard convention.

---

## Task 4: Weight the result

Each person in the sample stands for many people in Angola, and `weight_ind`
records how many. An unweighted rate describes the sample; a weighted rate
describes the country. Published statistics are always weighted.

In [6]:
def unemployment_rate(status, weights):
    """Unemployment as a percentage of the labour force."""
    unemployed = status == 'Unemployed'
    labour_force = status.isin(['Employed', 'Unemployed'])
    return weights[unemployed].sum() / weights[labour_force].sum() * 100


weight = df['weight_ind']
ones = pd.Series(1, index=df.index)

for name in ['lf_status_strict', 'lf_status_relaxed']:
    print(f'{name:20s} weighted: {unemployment_rate(df[name], weight):5.1f}%'
          f'   unweighted: {unemployment_rate(df[name], ones):5.1f}%')

lf_status_strict     weighted:  14.5%   unweighted:  14.4%
lf_status_relaxed    weighted:  31.4%   unweighted:  33.4%


In [7]:
in_labour_force = df['lf_status_strict'].isin(['Employed', 'Unemployed'])
participation = weight[in_labour_force].sum() / weight[working_age].sum() * 100
print(f'Labour force participation rate: {participation:.1f}%')
print(f'Weighted working age population: {weight[working_age].sum():,.0f}')

Labour force participation rate: 62.7%
Weighted working age population: 22,384,536


In [8]:
# Contributing family workers: employed under ICLS-19, excluded by the rule
# in Task 3. This is additive, it does not change lf_status_strict itself.
employed_with_family = employed | (df['worked_family_business'] == 1)
family_status = np.select(
    [working_age & employed_with_family, working_age & seeking & available],
    ['Employed', 'Unemployed'], default='Outside labour force')
family_status = pd.Series(family_status, index=df.index)

print('Working-age contributing family workers not otherwise employed:',
      (working_age & ~employed & (df['worked_family_business'] == 1)).sum())
print('Strict rate excluding them: %.1f%%' % unemployment_rate(df['lf_status_strict'], weight))
print('Strict rate including them: %.1f%%' % unemployment_rate(family_status, weight))

Working-age contributing family workers not otherwise employed: 3660
Strict rate excluding them: 14.5%
Strict rate including them: 11.2%


**Answers:**

- Strict unemployment is **14.5%** weighted. Relaxed unemployment is **31.4%**.
  Same data, same day, same people: a 17 point spread produced entirely by a
  definition.
- INE Angola publishes a figure around 29 to 30%, so the relaxed measure is the
  national headline. The strict measure is the internationally comparable one.
  Neither is wrong, and a table that does not say which it used is useless.
- The weights barely move this particular estimate (14.5% against 14.4%), because
  unemployment happens to be spread evenly across the weighting strata. That is
  luck, not a reason to skip them: the participation rate and every population
  total depend on them entirely.
- Which number goes in a press release? Whichever one the publication has always
  used, stated explicitly, with the other in a footnote. Switching silently
  between them is how a statistical office loses trust.
- A third defensible definition exists: count the 3,660 working-age people who
  work in a family business and are not otherwise employed as employed too, the
  ICLS-19 rule. That moves strict unemployment from 14.5% to **11.2%**. Three
  reasonable readers, three numbers, all from the same rows: the definition
  drives the headline more than the data does.

---

## Task 5: Binary flags with `np.where()` and `.loc[]`

`np.where()` is a vectorised if/else. `.loc[]` updates values that match a
condition, which is how you add a third state afterwards.

In [9]:
df['full_time'] = np.where(df['hours_usual'] >= 35, 'Full time', 'Part time')

# np.where has no idea what a missing value means: it lands in the else branch.
# Make the unknown explicit instead of letting it masquerade as part time.
df.loc[df['hours_usual'].isna(), 'full_time'] = 'Unknown'

df['full_time'].value_counts()

full_time
Unknown      41709
Full time     9054
Part time     2534
Name: count, dtype: int64

**Answers:**

- 9,054 full time, 2,534 part time, 41,709 unknown.
- Without the `.loc[]` line, all 41,709 people with no recorded hours would be
  labelled "Part time", and a headline about part time work would be off by a
  factor of sixteen. `np.where()` treats `NaN >= 35` as `False` and says nothing.
- The 41,709 are overwhelmingly people outside the labour force, who have no
  usual hours to report. Unknown is the honest label.

---

## Task 6: Dependent columns with `assign()`

`assign()` returns a new DataFrame, so it chains. A lambda inside it sees the
frame **as it is being built**, which is how the second column below can use the
first one created in the same call.

In [10]:
df = df.assign(
    job_tenure_years=lambda x: 2025 - x['job_start_year'],
    tenure_band=lambda x: np.select(
        [x['job_tenure_years'] < 1,
         x['job_tenure_years'] < 5,
         x['job_tenure_years'] >= 5],
        ['Under 1 year', '1 to 4 years', '5 years or more'],
        default='Unknown',
    ),
)
df['tenure_band'].value_counts()

tenure_band
Unknown            43373
5 years or more     5331
1 to 4 years        3324
Under 1 year        1269
Name: count, dtype: int64

**Answers:**

- 5,331 people have been in their job five years or more, 3,324 one to four
  years, 1,269 under a year, and 43,373 are Unknown.
- `tenure_band` must reference `lambda x: x['job_tenure_years']` rather than
  `df['job_tenure_years']`, because at that moment `df` is still the old frame and
  the column does not exist on it yet. `x` is the frame under construction.
- Unknown dominates because tenure only exists for people with a main job, and
  because `job_start_year` lost 1,673 sentinel values in 2.3. `np.select` routes
  every `NaN` to `default`, which is exactly what you want here.

---

## Task 7: Household size, without `groupby`

`hh_size_reported` was dropped in 2.3 because it was empty. Rebuild it from the
roster: count how many rows share each `household_id`, then map that count back
onto every person.

`value_counts()` comes from 2.1 and `.map()` from 2.4, so no new tool is needed.

In [11]:
df['hh_size'] = df['household_id'].map(df['household_id'].value_counts())

print('Mean household size per person:   ', round(df['hh_size'].mean(), 2))
print('Mean household size per household:',
      round(df.drop_duplicates('household_id')['hh_size'].mean(), 2))
df['hh_size'].describe()

Mean household size per person:    5.58
Mean household size per household: 4.09


count   53,297.00
mean         5.58
std          2.55
min          1.00
25%          4.00
50%          5.00
75%          7.00
max         20.00
Name: hh_size, dtype: double[pyarrow]

**Answers:**

- Per person the mean is **5.58**; per household it is **4.09**. Both are correct
  and they answer different questions.
- The gap exists because a household of 10 contributes 10 rows and a household of
  1 contributes one, so averaging over rows over-weights large households. "The
  average person lives in a household of 5.58" and "the average household has 4.09
  people" are both true statements.
- Publishing the per person figure as "average household size" is a classic error.
  Deduplicate to the household before averaging a household level attribute.

---

## Task 8: Custom logic with `apply()`

When built in operations cannot express the rule, `apply()` runs your own
function. It processes rows one at a time and is much slower than a vectorised
operation, so reach for it last, not first.

In [12]:
def hours_per_day(row):
    """Usual weekly hours spread over 7 days, or NaN when the input is unusable."""
    hours = row['hours_usual']
    if pd.isna(hours) or hours <= 0:
        return np.nan
    return round(hours / 7, 2)


df['hours_per_day'] = df.apply(hours_per_day, axis=1)
print('Mean hours per day:', round(df['hours_per_day'].mean(), 2))
df[['household_id', 'hours_usual', 'hours_per_day']].dropna().head()

Mean hours per day: 6.52


,household_id,hours_usual,hours_per_day
45,1000007,72.00,10.29
51,1000014,75.00,10.71
52,1000020,66.00,9.43
53,1000020,66.00,9.43
54,1000027,66.00,9.43


**Answers:**

- The mean is 6.52 hours a day across those who report any hours.
- `axis=1` passes a whole row, so the function can read several columns. Without
  it, `apply` would pass one column at a time.
- This particular calculation is a single division and would be far faster as
  `df['hours_usual'] / 7`. The guards are the only reason to use a function, and
  even they could be written as a vectorised `.where()`. On 53,000 rows the
  difference is invisible; on 5 million it is not.

---

## Task 9: Save the feature table

Cleaned data lives in `10_cleaned/`. Derived, analysis ready tables go in
`20_processed/`.

In [13]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)
out_path = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')

df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape)

Saved: ../../data/20_processed/angola_iea_2025q4_features.csv | (53297, 39)


In [14]:
check = pd.read_csv(out_path, dtype=STR_COLS)
print('Reloaded:', check.shape)
print('Columns added since the cleaned file:', check.shape[1] - 27)
check[['household_id', 'age_group', 'province_name',
       'lf_status_strict', 'lf_status_relaxed', 'hh_size']].head()

Reloaded: (53297, 39)
Columns added since the cleaned file: 12


,household_id,age_group,province_name,lf_status_strict,lf_status_relaxed,hh_size
0,10000003,Adult,Huila,Outside labour force,Outside labour force,4
1,10000003,Adult,Huila,Outside labour force,Outside labour force,4
2,10000003,Youth,Huila,Outside labour force,Outside labour force,4
3,10000003,Youth,Huila,Unemployed,Unemployed,4
4,10000009,Adult,Huila,Employed,Employed,6


**Answers:**

- 53,297 rows by 39 columns: 12 new columns on top of the 27 that came in.
- The vectorised ones (`pd.cut`, `.map`, `np.select`, `np.where`, the `assign`
  arithmetic) all operate on whole columns. Only `hours_per_day` uses `apply`
  with `axis=1`, so it is the first thing to rewrite if this ever runs on a
  census sized file.
- Row count is unchanged, which is the point: 2.3 decides what to remove, 2.4 only
  adds.